In [3]:
# ============================================================
# MORPHOLOGICAL TYPES FROM ANNUAL DEMs - BAÍA DA BABITONGA
# Separate processing for AOI01, AOI02, AOI03
# Google Colab
# ============================================================

# =========================
# 1) Install dependencies
# =========================
!pip -q install rasterio scikit-learn matplotlib pandas numpy scipy

# =========================
# 2) Imports
# =========================
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# =========================
# 3) Mount Google Drive
# =========================
drive.mount("/content/drive")

# ============================================================
# USER SETTINGS
# ============================================================

BASE_DIR = "/content/drive/Shareddrives/Batimetrias_Babitonga"
OUT_DIR = "/content/drive/MyDrive/babitonga_morph_types_by_aoi"
os.makedirs(OUT_DIR, exist_ok=True)

AOIS = ["01_AOI01", "02_AOI02", "03_AOI03"]

# Keep only pixels valid in at least this fraction of years
MIN_VALID_FRACTION = 0.5

# PCA settings
PCA_VARIANCE = 0.90
FIXED_N_PCS = None   # e.g. 3

# Candidate K values for clustering
K_CANDIDATES = [2, 3, 4, 5, 6]

RANDOM_STATE = 42

# If True, uses anomaly relative to mean DEM; recommended
USE_ANOMALY = True

# Reprojection/resampling method
RESAMPLING_METHOD = Resampling.bilinear

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def format_aoi_label(aoi_name):
    """
    Converte nomes de pasta como '01_AOI01' em rótulos limpos como 'AOI 1'
    """
    import re
    match = re.search(r'\d+', aoi_name)
    if match:
        return f"AOI {int(match.group())}"
    return aoi_name

def extract_year_from_path(path):
    """
    Extract year from folder/file path.
    Accepts 1900-2100.
    """
    matches = re.findall(r"(19\d{2}|20\d{2}|2100)", path)
    if not matches:
        return None
    return int(matches[-1])

def save_geotiff(out_path, arr, ref_meta):
    profile = {
        "driver": "GTiff",
        "height": ref_meta["height"],
        "width": ref_meta["width"],
        "count": 1,
        "dtype": "float32",
        "crs": ref_meta["crs"],
        "transform": ref_meta["transform"],
        "nodata": np.nan
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(arr.astype("float32"), 1)

def quicklook(arr, title, savepath, cmap="terrain", vmin=None, vmax=None, cbar_label=None):
    plt.figure(figsize=(8, 6))
    im = plt.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im)
    if cbar_label is not None:
        cbar.set_label(cbar_label)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.close()

def find_aoi_rasters(base_dir, aoi_name):
    """
    More robust search.
    Expected structure approximately:
    BASE_DIR/AOI_NAME/04_SUPERFICIES/YYYY/*.tif

    It accepts any tif/tiff inside each year folder and prefers files
    containing 'surface' in the name.
    """
    aoi_dir = os.path.join(base_dir, aoi_name)
    surf_dir = os.path.join(aoi_dir, "04_SUPERFICIES")

    print(f"\n[DEBUG] Searching in AOI dir: {aoi_dir}")
    print(f"[DEBUG] Exists AOI dir? {os.path.exists(aoi_dir)}")
    print(f"[DEBUG] Searching in surface dir: {surf_dir}")
    print(f"[DEBUG] Exists surface dir? {os.path.exists(surf_dir)}")

    if not os.path.exists(surf_dir):
        # fallback search for folder with similar name
        possible = glob.glob(os.path.join(aoi_dir, "*SUPERFIC*"))
        print(f"[DEBUG] Similar folders found: {possible}")
        if len(possible) > 0:
            surf_dir = possible[0]
            print(f"[DEBUG] Using fallback surface dir: {surf_dir}")
        else:
            return pd.DataFrame(columns=["year", "path"])

    year_dirs = sorted([d for d in glob.glob(os.path.join(surf_dir, "*")) if os.path.isdir(d)])
    print(f"[DEBUG] Year directories found: {len(year_dirs)}")

    records = []

    for yd in year_dirs:
        year = extract_year_from_path(yd)
        if year is None:
            continue

        tif_candidates = sorted(
            glob.glob(os.path.join(yd, "*.tif")) +
            glob.glob(os.path.join(yd, "*.TIF")) +
            glob.glob(os.path.join(yd, "*.tiff")) +
            glob.glob(os.path.join(yd, "*.TIFF"))
        )

        if len(tif_candidates) == 0:
            print(f"[DEBUG] No tif found in {yd}")
            continue

        # Prefer filenames containing "surface"
        preferred = [f for f in tif_candidates if "surface" in os.path.basename(f).lower()]
        chosen = preferred[0] if len(preferred) > 0 else tif_candidates[0]

        print(f"[DEBUG] Year {year}: chosen file -> {chosen}")
        records.append({"year": year, "path": chosen})

    if not records:
        return pd.DataFrame(columns=["year", "path"])

    df = (
        pd.DataFrame(records)
        .sort_values("year")
        .drop_duplicates(subset=["year"], keep="first")
        .reset_index(drop=True)
    )

    return df

def choose_reference_raster(file_list):
    """
    Choose the first readable raster as reference.
    If CRS is missing, assume EPSG:4326.
    """
    for f in file_list:
        try:
            with rasterio.open(f) as src:
                crs = src.crs if src.crs is not None else "EPSG:4326"
                return {
                    "path": f,
                    "crs": crs,
                    "transform": src.transform,
                    "width": src.width,
                    "height": src.height,
                    "dtype": src.dtypes[0],
                    "nodata": src.nodata
                }
        except Exception as e:
            print(f"[WARN] Could not open reference candidate {f}: {e}")
    raise RuntimeError("No readable raster found to define the reference grid.")

def read_and_match_to_reference(src_path, ref_meta, resampling_method=Resampling.bilinear):
    """
    Read raster, assign EPSG:4326 if CRS missing, and reproject to reference grid.
    Output is float32 with NaNs as nodata.
    """
    with rasterio.open(src_path) as src:
        arr = src.read(1).astype("float32")

        # Handle nodata
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        src_crs = src.crs
        if src_crs is None:
            src_crs = "EPSG:4326"

        # If same grid already, avoid reprojection
        same_grid = (
            src.width == ref_meta["width"] and
            src.height == ref_meta["height"] and
            src.transform == ref_meta["transform"] and
            str(src_crs) == str(ref_meta["crs"])
        )

        if same_grid:
            return arr

        dst = np.full((ref_meta["height"], ref_meta["width"]), np.nan, dtype="float32")

        reproject(
            source=arr,
            destination=dst,
            src_transform=src.transform,
            src_crs=src_crs,
            dst_transform=ref_meta["transform"],
            dst_crs=ref_meta["crs"],
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=resampling_method
        )

        return dst

def run_pca_clustering(X, years, variance_target=0.90, fixed_n_pcs=None, k_candidates=None, random_state=42):
    """
    X shape = (n_years, n_features)
    """
    if k_candidates is None:
        k_candidates = [2, 3, 4, 5, 6]

    # Fill remaining NaNs per pixel/feature
    imputer = SimpleImputer(strategy="mean")
    X_imp = imputer.fit_transform(X)

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)

    # PCA
    pca_full = PCA()
    X_pca_full = pca_full.fit_transform(X_scaled)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)

    if fixed_n_pcs is not None:
        n_pcs = fixed_n_pcs
    else:
        n_pcs = np.searchsorted(cumvar, variance_target) + 1

    X_pca = X_pca_full[:, :n_pcs]

    # Cluster selection
    scores = []
    models = {}

    max_k_allowed = max(2, len(years) - 1)
    k_candidates = [k for k in k_candidates if k < len(years) and k <= max_k_allowed]

    for k in k_candidates:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = km.fit_predict(X_pca)

        # silhouette requires at least 2 clusters and less than n_samples clusters
        try:
            score = silhouette_score(X_pca, labels)
        except:
            score = np.nan

        scores.append({"k": k, "silhouette": score})
        models[k] = (km, labels)

    df_scores = pd.DataFrame(scores)

    # fallback if silhouettes fail
    if df_scores["silhouette"].notna().any():
        best_k = int(df_scores.sort_values("silhouette", ascending=False).iloc[0]["k"])
    else:
        best_k = k_candidates[0]

    best_model, best_labels = models[best_k]

    df_pca = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(n_pcs)])
    df_pca.insert(0, "year", years)

    return {
        "imputer": imputer,
        "scaler": scaler,
        "pca_full": pca_full,
        "X_pca": X_pca,
        "df_pca": df_pca,
        "cumvar": cumvar,
        "n_pcs": n_pcs,
        "df_scores": df_scores,
        "best_k": best_k,
        "best_model": best_model,
        "best_labels": best_labels
    }

def process_aoi(base_dir, aoi_name, out_root):

    aoi_label = format_aoi_label(aoi_name)

    print("\n" + "="*70)
    print(f"PROCESSANDO {aoi_label}")
    print("="*70)

    aoi_out = os.path.join(out_root, aoi_name)
    os.makedirs(aoi_out, exist_ok=True)

    # -------------------------------------
    # 1) Find rasters
    # -------------------------------------
    df_files = find_aoi_rasters(base_dir, aoi_name)

    if df_files.empty:
        print(f"[WARN] No rasters found for {aoi_name}")
        return None

    print(df_files)
    print(f"Total years found: {len(df_files)}")

    # -------------------------------------
    # 2) Reference grid
    # -------------------------------------
    ref_meta = choose_reference_raster(df_files["path"].tolist())
    print("\nReference raster:")
    print(ref_meta)

    # -------------------------------------
    # 3) Read all years
    # -------------------------------------
    stack = []
    years = []
    used_files = []

    for _, row in df_files.iterrows():
        year = int(row["year"])
        path = row["path"]

        try:
            arr = read_and_match_to_reference(
                path,
                ref_meta,
                resampling_method=RESAMPLING_METHOD
            )

            if not np.isfinite(arr).any():
                print(f"[WARN] {aoi_name} {year}: raster has no valid pixels after loading/reprojection.")
                continue

            stack.append(arr)
            years.append(year)
            used_files.append(path)
            print(f"[OK] {aoi_name} {year}: {path}")

        except Exception as e:
            print(f"[FAIL] {aoi_name} {year}: {path} | {e}")

    if len(stack) < 3:
        print(f"[WARN] Too few valid rasters for {aoi_name}. Need at least 3.")
        return None

    stack = np.stack(stack, axis=0)   # (n_years, rows, cols)
    years = np.array(years)

    print(f"\nStack shape for {aoi_name}: {stack.shape}")

    # -------------------------------------
    # 4) Validity mask
    # -------------------------------------
    valid_fraction = np.mean(np.isfinite(stack), axis=0)
    keep_mask = valid_fraction >= MIN_VALID_FRACTION

    n_keep = int(keep_mask.sum())
    n_total = int(keep_mask.size)

    print(f"Pixels mantidos: {n_keep} / {n_total} ({100*n_keep/n_total:.2f}%)")

    if n_keep < 10:
        print(f"[WARN] Poucos pixels válidos para {aoi_name}.")
        return None

    # Save valid fraction map
    save_geotiff(os.path.join(aoi_out, "valid_fraction.tif"), valid_fraction.astype("float32"), ref_meta)
    quicklook(
        valid_fraction,
        title=f"{aoi_label} - Fração de observações válidas",
        savepath=os.path.join(aoi_out, "valid_fraction.png"),
        cmap="viridis",
        vmin=0,
        vmax=1,
        cbar_label="Fração de observações válidas"
    )

    # -------------------------------------
    # 5) Mean DEM and anomalies
    # -------------------------------------
    mean_dem = np.nanmean(stack, axis=0)
    save_geotiff(os.path.join(aoi_out, "mean_dem.tif"), mean_dem, ref_meta)
    quicklook(
        mean_dem,
        title=f"{aoi_label} - DEM médio",
        savepath=os.path.join(aoi_out, "mean_dem.png"),
        cmap="terrain",
        cbar_label="Elevação (m)"
    )

    if USE_ANOMALY:
        data_cube = stack - mean_dem
        analysis_name = "anomaly"
    else:
        data_cube = stack.copy()
        analysis_name = "raw_dem"

    # Matrix years x pixels
    X = data_cube[:, keep_mask]

    # -------------------------------------
    # 6) PCA + clustering
    # -------------------------------------
    results = run_pca_clustering(
        X=X,
        years=years,
        variance_target=PCA_VARIANCE,
        fixed_n_pcs=FIXED_N_PCS,
        k_candidates=K_CANDIDATES,
        random_state=RANDOM_STATE
    )

    df_result = results["df_pca"].copy()
    df_result["cluster"] = results["best_labels"] + 1
    df_result = df_result.sort_values("year").reset_index(drop=True)

    # Save tables
    df_result.to_csv(os.path.join(aoi_out, f"{aoi_name}_morph_types_by_year.csv"), index=False)
    df_files_used = pd.DataFrame({"year": years, "path": used_files})
    df_files_used.to_csv(os.path.join(aoi_out, f"{aoi_name}_files_used.csv"), index=False)
    results["df_scores"].to_csv(os.path.join(aoi_out, f"{aoi_name}_cluster_scores.csv"), index=False)

    print("\nResultados por ano:")
    print(df_result)

    # -------------------------------------
    # 7) Plots
    # -------------------------------------
    cumvar = results["cumvar"]
    n_pcs = results["n_pcs"]

    # Scree
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(cumvar) + 1), cumvar, marker="o")
    plt.axvline(n_pcs, linestyle="--")
    plt.xlabel("Número de componentes principais")
    plt.ylabel("Variância acumulada explicada")
    plt.title(f"{aoi_label} - Gráfico scree")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(aoi_out, f"{aoi_name}_scree_plot.png"), dpi=200, bbox_inches="tight")
    plt.close()

    # Cluster selection
    if not results["df_scores"].empty:
        plt.figure(figsize=(7, 4))
        plt.plot(results["df_scores"]["k"], results["df_scores"]["silhouette"], marker="o")
        plt.xlabel("Número de clusters (k)")
        plt.ylabel("Índice de silhueta")
        plt.title(f"{aoi_label} - Seleção do número de clusters")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(aoi_out, f"{aoi_name}_cluster_selection.png"), dpi=200, bbox_inches="tight")
        plt.close()

    # Temporal sequence of types
    plt.figure(figsize=(10, 4))
    plt.plot(df_result["year"], df_result["cluster"], marker="o")
    plt.xlabel("Ano")
    plt.ylabel("Tipo morfológico")
    plt.yticks(sorted(df_result["cluster"].unique()))
    plt.title(f"{aoi_label} - Sequência temporal dos tipos morfológicos")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(aoi_out, f"{aoi_name}_types_timeseries.png"), dpi=200, bbox_inches="tight")
    plt.close()

    # PC1 x PC2 scatter
    if "PC1" in df_result.columns and "PC2" in df_result.columns:
        plt.figure(figsize=(6, 5))
        scatter = plt.scatter(df_result["PC1"], df_result["PC2"], c=df_result["cluster"])
        for _, row in df_result.iterrows():
            plt.text(row["PC1"], row["PC2"], str(int(row["year"])), fontsize=8)
        plt.xlabel("CP1")
        plt.ylabel("CP2")
        plt.title(f"{aoi_label} - Espaço dos componentes principais")
        plt.tight_layout()
        plt.savefig(os.path.join(aoi_out, f"{aoi_name}_pca_scatter.png"), dpi=200, bbox_inches="tight")
        plt.close()

    # -------------------------------------
    # 8) Mean anomaly per cluster
    # -------------------------------------
    for c in sorted(df_result["cluster"].unique()):
        idx = np.where(df_result["cluster"].values == c)[0]
        cluster_mean = np.nanmean(data_cube[idx, :, :], axis=0)

        tif_path = os.path.join(aoi_out, f"{aoi_name}_type_{c}_{analysis_name}_mean.tif")
        png_path = os.path.join(aoi_out, f"{aoi_name}_type_{c}_{analysis_name}_mean.png")

        save_geotiff(tif_path, cluster_mean, ref_meta)

        vmax = np.nanpercentile(np.abs(cluster_mean[np.isfinite(cluster_mean)]), 98) if np.isfinite(cluster_mean).any() else 1
        if vmax == 0:
            vmax = 1

        if USE_ANOMALY:
            titulo_tipo = f"{aoi_label} - Tipo {c}: anomalia média"
            rotulo_cbar = "Anomalia de elevação (m)"
        else:
            titulo_tipo = f"{aoi_label} - Tipo {c}: DEM médio"
            rotulo_cbar = "Elevação (m)"

        quicklook(
            cluster_mean,
            title=titulo_tipo,
            savepath=png_path,
            cmap="RdBu_r" if USE_ANOMALY else "terrain",
            vmin=-vmax if USE_ANOMALY else None,
            vmax=vmax if USE_ANOMALY else None,
            cbar_label=rotulo_cbar
        )

    # -------------------------------------
    # 9) EOF maps
    # -------------------------------------
    components = results["pca_full"].components_[:n_pcs, :]

    for i in range(n_pcs):
        eof_map = np.full(keep_mask.shape, np.nan, dtype="float32")
        eof_map[keep_mask] = components[i, :]

        tif_path = os.path.join(aoi_out, f"{aoi_name}_EOF_{i+1}.tif")
        png_path = os.path.join(aoi_out, f"{aoi_name}_EOF_{i+1}.png")

        save_geotiff(tif_path, eof_map, ref_meta)

        valid_vals = eof_map[np.isfinite(eof_map)]
        vmax = np.nanpercentile(np.abs(valid_vals), 98) if valid_vals.size > 0 else 1
        if vmax == 0:
            vmax = 1

        quicklook(
            eof_map,
            title=f"{aoi_label} - EOF {i+1}",
            savepath=png_path,
            cmap="RdBu_r",
            vmin=-vmax,
            vmax=vmax,
            cbar_label=f"Carga espacial da EOF {i+1}"
        )

    # -------------------------------------
    # 10) Summary text
    # -------------------------------------
    with open(os.path.join(aoi_out, f"{aoi_name}_summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"AOI: {aoi_name}\n")
        f.write("="*60 + "\n")
        f.write(f"Modo de análise: {analysis_name}\n")
        f.write(f"Número de anos utilizados: {len(years)}\n")
        f.write(f"Anos: {years.tolist()}\n")
        f.write(f"Pixels mantidos: {n_keep} / {n_total} ({100*n_keep/n_total:.2f}%)\n")
        f.write(f"Fração mínima de dados válidos: {MIN_VALID_FRACTION}\n")
        f.write(f"Número de componentes principais selecionados: {n_pcs}\n")
        f.write(f"Variância acumulada explicada: {cumvar[n_pcs-1]:.4f}\n")
        f.write(f"Melhor k: {results['best_k']}\n\n")

        f.write("Variância explicada por componente principal selecionado:\n")
        for i, val in enumerate(results["pca_full"].explained_variance_ratio_[:n_pcs], start=1):
            f.write(f"CP{i}: {val:.4f}\n")

        f.write("\nÍndices de silhueta:\n")
        if not results["df_scores"].empty:
            for _, row in results["df_scores"].sort_values("k").iterrows():
                f.write(f"k={int(row['k'])}: {row['silhouette']}\n")

        f.write("\nTipo morfológico por ano:\n")
        f.write(df_result.to_string(index=False))
        f.write("\n")

    return {
        "aoi": aoi_name,
        "years": years,
        "df_result": df_result,
        "best_k": results["best_k"],
        "n_pcs": n_pcs,
        "cumvar": cumvar[n_pcs-1],
        "pixels_kept": n_keep,
        "total_pixels": n_total
    }

# ============================================================
# MAIN LOOP
# ============================================================

all_summaries = []

for aoi in AOIS:
    result = process_aoi(BASE_DIR, aoi, OUT_DIR)
    if result is not None:
        all_summaries.append(result)

# ============================================================
# GLOBAL SUMMARY
# ============================================================

if all_summaries:
    summary_rows = []
    for r in all_summaries:
        summary_rows.append({
            "AOI": r["aoi"],
            "n_years": len(r["years"]),
            "year_min": int(np.min(r["years"])),
            "year_max": int(np.max(r["years"])),
            "best_k": r["best_k"],
            "n_pcs": r["n_pcs"],
            "cum_explained_variance": r["cumvar"],
            "pixels_kept": r["pixels_kept"],
            "total_pixels": r["total_pixels"],
            "pct_pixels_kept": 100 * r["pixels_kept"] / r["total_pixels"]
        })

    df_global = pd.DataFrame(summary_rows)
    print("\nRESUMO GERAL")
    print(df_global)

    df_global.to_csv(os.path.join(OUT_DIR, "global_summary.csv"), index=False)

    with open(os.path.join(OUT_DIR, "README_results.txt"), "w", encoding="utf-8") as f:
        f.write("Babitonga morphological types analysis - outputs\n")
        f.write("="*70 + "\n\n")
        f.write("For each AOI folder, the main outputs are:\n")
        f.write("- mean_dem.tif / .png\n")
        f.write("- valid_fraction.tif / .png\n")
        f.write("- AOI_morph_types_by_year.csv\n")
        f.write("- AOI_cluster_scores.csv\n")
        f.write("- AOI_types_timeseries.png\n")
        f.write("- AOI_pca_scatter.png\n")
        f.write("- AOI_EOF_*.tif / .png\n")
        f.write("- AOI_type_*_anomaly_mean.tif / .png\n")
        f.write("- AOI_summary.txt\n")

print(f"\nProcessamento concluído. Resultados salvos em: {OUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

PROCESSANDO AOI 1

[DEBUG] Searching in AOI dir: /content/drive/Shareddrives/Batimetrias_Babitonga/01_AOI01
[DEBUG] Exists AOI dir? True
[DEBUG] Searching in surface dir: /content/drive/Shareddrives/Batimetrias_Babitonga/01_AOI01/04_SUPERFICIES
[DEBUG] Exists surface dir? True
[DEBUG] Year directories found: 33
[DEBUG] Year 1985: chosen file -> /content/drive/Shareddrives/Batimetrias_Babitonga/01_AOI01/04_SUPERFICIES/1985/sup_delta_1985.tif
[DEBUG] Year 1986: chosen file -> /content/drive/Shareddrives/Batimetrias_Babitonga/01_AOI01/04_SUPERFICIES/1986/sup_delta_1986_v2.tif
[DEBUG] Year 1987: chosen file -> /content/drive/Shareddrives/Batimetrias_Babitonga/01_AOI01/04_SUPERFICIES/1987/sup_delta_1987.tif
[DEBUG] Year 1988: chosen file -> /content/drive/Shareddrives/Batimetrias_Babitonga/01_AOI01/04_SUPERFICIES/1988/sup_delta_1988.tif
[DEBUG] Year 1991: chosen 